# MCXN9xx Device HSM (DEVHSM) Provisioning

This notebook describes how to perform Device HSM (DEVHSM) trust provisioning on the MCXN9xx device family (e.g. `mcxn946`/`mcxn947`) using the SPSDK *nxpdevhsm*, *pfr* and *nxpimage* applications.

DEVHSM lets you securely provision a device in a single step by building a provisioning Secure Binary (SB) file whose encrypted blocks are generated **on the device itself** using the on-chip EdgeLock subsystem, and then merged on the host PC. In this example the provisioning SB file:

- injects the customer master symmetric key (**CUST_MK_SK / SB3KDK**) into the device keystore in a secure way, and
- loads the chip configuration (**CMPA/CFPA**), which carries the Root of Trust (RKTH) and secure boot settings.

Once provisioned, the device only accepts SB3.1 files that are signed against the Root of Trust in the CMPA and encrypted with the CUST_MK_SK stored in the keystore (see the [SB3.1 example](../../sb/mcxn9xx/mcxn9xx_sb31.ipynb)).

<div class="alert alert-block alert-warning">
<b>WARNING:</b> The keys and configuration used in this example are meant only for demonstration purposes and must never be used in production. Some DEVHSM steps (e.g. advancing the device life cycle) are irreversible.
</div>

## 1. Prerequisites

- SPSDK is needed with examples extension. `pip install spsdk[examples]` (Please refer to the [installation](../../_knowledge_base/installation_guide.rst) documentation.)
- This example uses an MCXN9xx board (e.g. FRDM-MCXN947) connected via USB (ISP mode) or UART.

<img src="../../_data/img/boards/frdm-mcxn947.png" alt="frdm-mcxn947" width="400">

In [1]:
# Initialization cell
from spsdk.utils.jupyter_utils import YamlDiffWidget

# This env variable sets colored logger output to STDOUT
%env JUPYTER_SPSDK=1
# Set a magic for command execution and echo
%alias execute echo %l && %l
%alias_magic ! execute

env: JUPYTER_SPSDK=1
Created `%!` as an alias for `%execute`.


In [2]:
FAMILY = "mcxn946"  # change this to your MCXN9xx family (e.g. mcxn947)
WORKSPACE = "workspace/"  # change this to path to your workspace
INPUTS = "inputs/"  # input configurations and keys
KEYS = "../../_data/keys/ecc256/"  # shared example keys
VERBOSITY = (
    ""  # verbosity of commands, might be -v or -vv for debug or blank for no additional info
)

## 2. Root of Trust and CUST_MK_SK

Two secrets are involved in DEVHSM provisioning:

- The **Root of Trust** keys. Their table hash (RKTH) is stored in the CMPA and defines which SB3.1 files the device accepts. In this example we reuse the shared ECC-256 example key-pairs from the `_data/keys/ecc256` folder; in a real scenario you would generate your own keys using the *nxpcrypto* application (refer to [How-to-get-keys-using-nxpcrypto](../../crypto/keys.ipynb)).
- The **CUST_MK_SK** (Part Common Key / SB3KDK), a 256-bit customer master symmetric key used to derive the SB3.1 block encryption keys. It is provided here as a plain hex string in [cust_mk_sk.txt](./inputs/cust_mk_sk.txt).

## 3. Certificate block

The certificate block holds the Root of Trust information. It is referenced when generating the CMPA binary so that the correct RKTH is written into the device. A ready-made configuration is provided in [cert_block_mcxn9xx.yaml](./inputs/cert_block_mcxn9xx.yaml). To learn how to create a certificate block from scratch, check the dedicated notebook (see [How-to-get-cert-block](../../certificate_block/cert_block_v2_1.ipynb)).

## 4. Generate CMPA and CFPA binaries

The device configuration areas CMPA (Customer Manufacturing Programmable Area) and CFPA (Customer In-Field Programmable Area) are generated with the *pfr* application. The CMPA export takes the certificate block (`-e` option) to embed the RKTH. Ready-made minimal configurations are provided in the inputs folder; extend them to enable secure boot, restrict debug, disable ISP, etc.

In [3]:
import os

CMPA_CONFIG = INPUTS + "cmpa_mcxn9xx.yaml"
CFPA_CONFIG = INPUTS + "cfpa_mcxn9xx.yaml"
CERT_BLOCK_CONFIG = INPUTS + "cert_block_mcxn9xx.yaml"
CMPA_BIN = WORKSPACE + "cmpa.bin"
CFPA_BIN = WORKSPACE + "cfpa.bin"

%! pfr $VERBOSITY export -c $CMPA_CONFIG -e $CERT_BLOCK_CONFIG -o $CMPA_BIN
%! pfr $VERBOSITY export -c $CFPA_CONFIG -o $CFPA_BIN
assert os.path.exists(CMPA_BIN)
assert os.path.exists(CFPA_BIN)

pfr  export -c inputs/cmpa_mcxn9xx.yaml -e inputs/cert_block_mcxn9xx.yaml -o workspace/cmpa.bin 


Success. (CMPA binary has been generated and stored into workspace/cmpa.bin)


pfr  export -c inputs/cfpa_mcxn9xx.yaml -o workspace/cfpa.bin 
Success. (CFPA binary has been generated and stored into workspace/cfpa.bin)


## 5. Prepare the DEVHSM configuration file

The DEVHSM provisioning SB file is described by a YAML configuration, which can be created with `nxpdevhsm get-template` and then customized. Below we compare the generated template with the ready-made user configuration ([devhsm_config.yaml](./inputs/devhsm_config.yaml)) to highlight the changes we made. The configuration loads the CMPA/CFPA binaries and transfers the CUST_MK_SK into the keystore; the key blob load command is injected automatically from `containerKeyBlobEncryptionKey`.

In [4]:
# Get difference of template and user YAML configuration
YamlDiffWidget("inputs/mcxn9xx_devhsm.diffc").html

nxpdevhsm get-template -f mcxn946 -o workspace/devhsm_template.yaml --force 
The DevHsm template for mcxn946, Revision: latest has been saved into workspace/devhsm_template.yaml YAML file


## 6. Device preparation

Put the board into In-System Programming (ISP) mode and use *nxpdevscan* to check that the device is detected by the PC.

In [5]:
# check if the device is connected and detected by the PC
%! nxpdevscan

nxpdevscan 
Scanning USB devices...
╔══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║                                        Connected NXP USB Devices  (1 device)                                         ║
╠══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╣
║  VID       : 0x1fc9                                                                                                  ║
║  PID       : 0x014f                                                                                                  ║
║  Product   : USB COMPOSITE DEVICE                                                                                    ║
║  Name      : kw43b43zc2 | mcxw706d | kw43b43zc3 | mcxn557t | mcxn947t | mcxn246t | mcxn946t | mcxn246 | mcxn947 | mcx║
║  Path      : HID\VID_1FC9&PID_014F\6&159A9FAD&1&0000                                                               

In [6]:
# Select the interface to the target. USB (default MCXN9xx ISP VID:PID) or UART.
USB = "-u 0x1fc9,0x014f"  # change to your device, or use e.g. "-p com10" for UART

## 7. Generate and load the DEVHSM provisioning SB file

The `nxpdevhsm generate` command connects to the device, uses the on-chip EdgeLock subsystem to encrypt and sign the SB blocks, and merges the resulting provisioning SB file on the host. The CUST_MK_SK, CMPA and CFPA are injected into the device in a secure, encrypted way.

By default the device is reset after the operation (controlled by `finalReset` in the configuration). Once reset, the provisioned chip only accepts authenticated and encrypted SB3.1 files.

In [7]:
DEVHSM_CONFIG = INPUTS + "devhsm_config.yaml"
%! nxpdevhsm $VERBOSITY generate $USB -c $DEVHSM_CONFIG

nxpdevhsm  generate -u 0x1fc9,0x014f -c inputs/devhsm_config.yaml 
 1: Initial target reset is disabled 
 2: Generating OEM master share.
 3: Generating 48 bytes FW signing keys.
 4: Generating 48 bytes FW encryption keys.
 5: Creating template un-encrypted SB3 header and data blobs.
 5.1: Creating template SB3 header.
 5.2: Creating un-encrypted SB3 data.
 5.3 Looking for plaintext keys to wrap
 5.4 Handling CUST_MK_SK/SBKEK keyblob
 5.4.1 Injecting wrapped key
 5.4.2: Wrapping CUST_MK_SK key.
 5.4.3: Adding wrapped CUST_MK_SK LoadKeyBlob command into SB file.
 6: Encrypting SB3 data on device
 6.1: Enriching encrypted SB3 data by mandatory hashes.
 6.2: Creating dummy certificate block.
 6.3: Updating SB3 header by valid values.
 6.4: Preparing SB3 manifest to sign.
 7: Creating SB3 manifest signature on device.
 8: Composing final SB3 file.
 9: Resetting the target device
Final SB file has been written: workspace/mcxn9xx_devhsm.sb


The generated provisioning SB file can also be applied later with *blhost* if the device is not automatically reset:

In [8]:
DEVHSM_SB = WORKSPACE + "mcxn9xx_devhsm.sb"
%! blhost $USB receive-sb-file $DEVHSM_SB

blhost -u 0x1fc9,0x014f receive-sb-file workspace/mcxn9xx_devhsm.sb 
Response status = 0 (0x0) Success.
